In [52]:
"""
Terra-Luna Crisis Analysis - Production Version
Integrates with your exact data loading method from ZIP file

Dataset: token_transfers_V3.0.0.csv from ERC20-stablecoins.zip
"""

import pandas as pd
import numpy as np
import zipfile
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import your visualization tools
from src.DatasetManager import DatasetManager
from src.Visualiser import Visualiser


# ============================================================================
# CONFIGURATION
# ============================================================================

# File paths
ZIP_PATH = "datasets/ERC20-stablecoins.zip"
CSV_FILE = "token_transfers_V3.0.0.csv"

# USDT contract address (to filter if needed)
USDT_CONTRACT = "0xa47c8bf37f92abed4a126bda807a7b7498661acd"

# USDT has 6 decimal places
USDT_DECIMALS = 1e6

# Analysis period
ANALYSIS_START = '2022-04-20'
ANALYSIS_END = '2022-05-30'
CRISIS_START = '2022-05-03'
CRISIS_END = '2022-05-20'

# Analysis parameters
LARGE_TX_PERCENTILE = 95
STRESS_VOLUME_PERCENTILE = 90

In [53]:
# ============================================================================
# LOAD DATA
# ============================================================================

print("Loading data from ZIP file...")
print(f"ZIP: {ZIP_PATH}")
print(f"CSV: {CSV_FILE}")

# Load data using your method
# NOTE: Remove 'nrows=5' for full analysis - that's just for testing!
with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(CSV_FILE) as f:
        # For full analysis, remove nrows parameter:
        # token_df = pd.read_csv(f, encoding='latin-1')
        
        # For testing with sample:
        token_df = pd.read_csv(f, encoding='latin-1')  # Adjust as needed


Loading data from ZIP file...
ZIP: datasets/ERC20-stablecoins.zip
CSV: token_transfers_V3.0.0.csv


In [54]:

print(f"✓ Loaded {len(token_df):,} transactions")
print(f"\nColumns: {list(token_df.columns)}")
print(f"\nFirst few rows:")
print(token_df.head())
# Check data structure
print(f"\nData types:")
print(token_df.dtypes)

✓ Loaded 36,723,655 transactions

Columns: ['block_number', 'transaction_index', 'from_address', 'to_address', 'time_stamp', 'contract_address', 'value']

First few rows:
   block_number  transaction_index  \
0      14500001                 44   
1      14500001                 62   
2      14500001                 64   
3      14500001                 68   
4      14500003                  0   

                                 from_address  \
0  0x27cbb0e6885ccb1db2dab7c2314131c94795fbef   
1  0x7938b1b2f2d2ec6cde2db46fcb11d824f32eae54   
2  0x4593e0fb8dcc65cd24c7f99ee64da2627e90f998   
3  0x3cd751e6b0078be393132286c442345e5dc49699   
4  0xffec0067f5a79cff07527f63d83dd5462ccf8ba4   

                                   to_address  time_stamp  \
0  0x8426a27add8dca73548f012d92c7f8f4bbd42a3e  1648811421   
1  0xffabc91efaf240a48fe2b31d3599925d3504c3df  1648811421   
2  0x97138e4cb36db0185236c3d74cb39fb51cb3228b  1648811421   
3  0xf4fc2e12974cc3b4d8192722430c571968025d4f  1648811421   


In [55]:
# ============================================================================
# DATA PREPROCESSING
# ============================================================================

print("\n" + "="*80)
print("PREPROCESSING DATA")
print("="*80)

# 1. Convert value to number of coins 
token_df['amount'] = token_df['value'] 
print(f"✓ Converted value to USD (divided by {USDT_DECIMALS:,.0f})")

# Sanity check
print(f"  Sample values:")
print(f"  - Raw value: {token_df['value'].iloc[0]:,.0f} → Tokens:{token_df['amount'].iloc[0]:,.6f}")
print(f"  - Median raw: {token_df['value'].median():,.0f} → Tokens:{token_df['amount'].median():,.2f}")
print(f"  - Max raw: {token_df['value'].max():,.0f} → Tokens:{token_df['amount'].max():,.2f}")

# 2. Convert timestamp to datetime
token_df['Date'] = pd.to_datetime(token_df['time_stamp'], unit='s')
print(f"✓ Converted timestamps to datetime")
print(f"  Date range: {token_df['Date'].min()} to {token_df['Date'].max()}")

# 3. Create time-based features
token_df['Hour'] = token_df['Date'].dt.floor('h')
token_df['Day'] = token_df['Date'].dt.date
token_df['DayOfWeek'] = token_df['Date'].dt.day_name()
print(f"✓ Created time features (Hour, Day, DayOfWeek)")

# 4. Sort by timestamp (CRITICAL for rolling calculations)
token_df = token_df.sort_values('Date').reset_index(drop=True)
print(f"✓ Sorted by date")

# 5. Filter to analysis period
mask = (token_df['Date'] >= ANALYSIS_START) & (token_df['Date'] <= ANALYSIS_END)
token_df_filtered = token_df[mask].copy()
print(f"✓ Filtered to analysis period: {ANALYSIS_START} to {ANALYSIS_END}")
print(f"  Transactions in period: {len(token_df_filtered):,}")

# Check if we have data in the crisis period
crisis_mask = (token_df_filtered['Date'] >= CRISIS_START) & (token_df_filtered['Date'] <= CRISIS_END)
crisis_txs = crisis_mask.sum()
print(f"  Transactions during crisis ({CRISIS_START} to {CRISIS_END}): {crisis_txs:,}")

if crisis_txs == 0:
    print("\n⚠️  WARNING: No transactions found in crisis period!")
    print("    This might mean:")
    print("    1. You're using nrows limit that's too small")
    print("    2. Dataset doesn't cover May 2022")
    print("    3. Need to filter by contract_address for USDT only")
    print("\n    Continuing with available data...\n")

# Use filtered data for analysis
token_df = token_df_filtered # Analysis Start - End Timeframe


PREPROCESSING DATA
✓ Converted value to USD (divided by 1,000,000)
  Sample values:
  - Raw value: 800 → Tokens:800.000000
  - Median raw: 1,030 → Tokens:1,030.48
  - Max raw: 676,792,330,237 → Tokens:676,792,330,236.96
✓ Converted timestamps to datetime
  Date range: 2022-04-01 11:10:21 to 2022-11-01 04:53:59
✓ Created time features (Hour, Day, DayOfWeek)
✓ Sorted by date
✓ Filtered to analysis period: 2022-04-20 to 2022-05-30
  Transactions in period: 7,014,365
  Transactions during crisis (2022-05-03 to 2022-05-20): 3,707,062


In [56]:
# ============================================================================
# EXPLORATORY DATA ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

# Basic statistics
print(f"\nToken Transaction Amount Statistics:")
print(token_df['amount'].describe())

print(f"\nDate Distribution:")
print(token_df['Date'].dt.date.value_counts().sort_index().head(5))

print(f"\nUnique Participants:")
print(f"  Unique senders: {token_df['from_address'].nunique():,}")
print(f"  Unique receivers: {token_df['to_address'].nunique():,}")

# Check for contract filtering
if 'contract_address' in token_df.columns:
    print(f"\nContract Addresses in Data:")
    print(token_df['contract_address'].value_counts())
    
    # Filter to USDT only if multiple contracts present
    if token_df['contract_address'].nunique() > 1:
        print(f"\n⚠️  Multiple contracts detected! Filtering to USDT only...")
        token_df = token_df[token_df['contract_address'].str.lower() == USDT_CONTRACT.lower()].copy()
        print(f"  USDT transactions: {len(token_df):,}")


EXPLORATORY DATA ANALYSIS

Token Transaction Amount Statistics:
count    7.014365e+06
mean     2.510332e+06
std      6.035777e+08
min      0.000000e+00
25%      3.603142e+02
50%      2.000000e+03
75%      1.500000e+04
max      6.767923e+11
Name: amount, dtype: float64

Date Distribution:
Date
2022-04-20    141338
2022-04-21    149541
2022-04-22    140405
2022-04-23    113734
2022-04-24    103898
Name: count, dtype: int64

Unique Participants:
  Unique senders: 1,503,441
  Unique receivers: 1,603,212

Contract Addresses in Data:
contract_address
0xdac17f958d2ee523a2206206994597c13d831ec7    3577657
0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48    2673502
0x6b175474e89094c44da98b954eedeac495271d0f     394702
0xd2877702675e6ceb975b4a1dff9fb7baf4c91ea9     225161
0xa47c8bf37f92abed4a126bda807a7b7498661acd     128601
0x8e870d67f660d95d5be530380d0ec0bd388289e1      14742
Name: count, dtype: int64

⚠️  Multiple contracts detected! Filtering to USDT only...
  USDT transactions: 128,601


In [57]:
# ============================================================================
# CALCULATE ROLLING METRICS (24-HOUR WINDOWS)
# ============================================================================

print("\n" + "="*80)
print("CALCULATING ROLLING METRICS (24H WINDOWS)")
print("="*80)

# Set Date as index for rolling operations
df_indexed = token_df.set_index('Date').sort_index()
df_indexed.index = pd.DatetimeIndex(df_indexed.index)
print("Calculating volume metrics...")
# 1. Total volume (24h rolling sum)
df_indexed['Volume_24h'] = df_indexed['amount'].rolling(
    window='24h', min_periods=1
).sum()

print("Calculating transaction count...")
# 2. Transaction count (24h rolling count)
df_indexed['TxCount_24h'] = df_indexed['amount'].rolling(
    window='24h', min_periods=1
).count()

print("Calculating average transaction size...")
# 3. Average transaction size (whale indicator)
df_indexed['AvgTxSize_24h'] = df_indexed['amount'].rolling(
    window='24h', min_periods=1
).mean()

# 4. Median transaction size (robust to outliers)
df_indexed['MedianTxSize_24h'] = df_indexed['amount'].rolling(
    window='24h', min_periods=1
).median()

print("Calculating volatility metrics...")
# 5. Transaction volatility (market stress indicator)
df_indexed['TxVolatility_24h'] = df_indexed['amount'].rolling(
    window='24h', min_periods=1
).std()

print("Identifying large transactions...")
# 6. Large transaction indicator
large_tx_threshold = df_indexed['amount'].quantile(LARGE_TX_PERCENTILE / 100)
df_indexed['is_large_tx'] = (df_indexed['amount'] >= large_tx_threshold).astype(int)
df_indexed['LargeTxCount_24h'] = df_indexed['is_large_tx'].rolling(
    window='24h', min_periods=1
).sum()

print(f"✓ Large transaction threshold (95th percentile): ${large_tx_threshold:,.2f}")
print(f"✓ Number of large transactions: {df_indexed['is_large_tx'].sum():,}")

# Reset index
token_df = df_indexed.reset_index()

print("✓ Rolling metrics calculated")


CALCULATING ROLLING METRICS (24H WINDOWS)
Calculating volume metrics...
Calculating transaction count...
Calculating average transaction size...
Calculating volatility metrics...
Identifying large transactions...
✓ Large transaction threshold (95th percentile): $360,788.72
✓ Number of large transactions: 6,432
✓ Rolling metrics calculated


In [58]:
# ============================================================================
# CALCULATE HOURLY AGGREGATES
# ============================================================================

print("\n" + "="*80)
print("CALCULATING HOURLY AGGREGATES")
print("="*80)

hourly = token_df.groupby('Hour').agg({
    'amount': ['sum', 'mean', 'median', 'std', 'count'],
    'from_address': 'nunique',
    'to_address': 'nunique',
    'is_large_tx': 'sum'
}).reset_index()

# Flatten multi-index columns
hourly.columns = ['Hour', 'Total_Volume', 'Avg_Tx', 'Median_Tx', 'Tx_Std', 
                  'Tx_Count', 'Unique_Senders', 'Unique_Receivers', 'Large_Tx_Count']

# Calculate derived metrics
hourly['Volume_Per_Sender'] = hourly['Total_Volume'] / hourly['Unique_Senders']
hourly['Tx_Per_Sender'] = hourly['Tx_Count'] / hourly['Unique_Senders']

# Hour-over-hour changes
hourly['Volume_Change_Pct'] = hourly['Total_Volume'].pct_change() * 100
hourly['TxCount_Change_Pct'] = hourly['Tx_Count'].pct_change() * 100

# Identify stress periods
volume_threshold = hourly['Total_Volume'].quantile(STRESS_VOLUME_PERCENTILE / 100)
count_threshold = hourly['Tx_Count'].quantile(STRESS_VOLUME_PERCENTILE / 100)
hourly['is_stress_period'] = (
    (hourly['Total_Volume'] >= volume_threshold) & 
    (hourly['Tx_Count'] >= count_threshold)
)

stress_hours = hourly['is_stress_period'].sum()
print(f"✓ Hourly aggregates calculated: {len(hourly)} hours")
print(f"✓ Stress periods identified: {stress_hours} hours ({stress_hours/len(hourly)*100:.1f}%)")


CALCULATING HOURLY AGGREGATES
✓ Hourly aggregates calculated: 958 hours
✓ Stress periods identified: 61 hours (6.4%)


In [81]:
# recompute daily from current token_df in memory
daily = token_df.groupby('Day').agg(
    Daily_Volume=('amount', 'sum'),
    Unique_Senders=('from_address', 'nunique'),
    Unique_Receivers=('to_address', 'nunique'),
    Tx_Count=('amount', 'count')
).reset_index()
daily.columns = ['Day', 'Daily_Volume', 'Unique_Senders', 'Unique_Receivers','Tx_Count']

# recompute HHI
hhi_data = []
for day in token_df['Day'].unique():
    day_txs = token_df[token_df['Day'] == day]
    sender_volumes = day_txs.groupby('from_address')['amount'].sum()
    total_volume = sender_volumes.sum()
    hhi = (sender_volumes / total_volume).pow(2).sum() if total_volume > 0 else float('nan')
    hhi_data.append({'Day': day, 'HHI_Senders': hhi})

daily = daily.merge(pd.DataFrame(hhi_data), on='Day', how='left')
daily['Day'] = pd.to_datetime(daily['Day'])


In [82]:
# ============================================================================
# KEY STATISTICS FOR SECTION A, QUESTION 1
# ============================================================================

print("\n" + "="*80)
print("SECTION A, QUESTION 1: KEY FINDINGS")
print("When the Peg Breaks: Onset and Spread")
print("="*80)
# Baseline period (pre-crisis)
baseline_data = token_df[token_df['Date'] < CRISIS_START]
crisis_data = token_df[(token_df['Date'] >= CRISIS_START) & (token_df['Date'] <= CRISIS_END)]

if len(baseline_data) > 0 and len(crisis_data) > 0:
    
    # Volume analysis
    baseline_vol = baseline_data.groupby('Day')['amount'].sum().mean()
    crisis_vol = crisis_data.groupby('Day')['amount'].sum().mean()
    peak_vol_hourly = hourly['Total_Volume'].max()
    peak_hour = hourly.loc[hourly['Total_Volume'].idxmax(), 'Hour']
    
    print(f"\n📊 VOLUME DYNAMICS:")
    print(f"   Baseline avg daily volume:     ${baseline_vol:,.0f}")
    print(f"   Crisis avg daily volume:       ${crisis_vol:,.0f}")
    print(f"   Increase:                      {(crisis_vol/baseline_vol - 1)*100:.1f}%")
    print(f"   Peak hourly volume:            ${peak_vol_hourly:,.0f}")
    print(f"   Peak occurred:                 {peak_hour}")
    
    # Transaction count
    baseline_count = baseline_data.groupby('Day').size().mean()
    crisis_count = crisis_data.groupby('Day').size().mean()
    peak_count = hourly['Tx_Count'].max()
    peak_count_hour = hourly.loc[hourly['Tx_Count'].idxmax(), 'Hour']
    
    print(f"\n🔄 TRANSACTION ACTIVITY:")
    print(f"   Baseline avg daily txs:        {baseline_count:.0f}")
    print(f"   Crisis avg daily txs:          {crisis_count:.0f}")
    print(f"   Increase:                      {(crisis_count/baseline_count - 1)*100:.1f}%")
    print(f"   Peak hourly txs:               {peak_count:,}")
    print(f"   Peak occurred:                 {peak_count_hour}")
    
    # Large transaction analysis
    baseline_large = baseline_data[baseline_data['is_large_tx'] == 1].groupby('Day').size().mean()
    crisis_large = crisis_data[crisis_data['is_large_tx'] == 1].groupby('Day').size().mean()
    
    print(f"\n🐋 WHALE ACTIVITY (>{LARGE_TX_PERCENTILE}th percentile):")
    print(f"   Baseline avg daily large txs:  {baseline_large:.1f}")
    print(f"   Crisis avg daily large txs:    {crisis_large:.1f}")
    print(f"   Increase:                      {(crisis_large/baseline_large - 1)*100:.1f}%")
    print(f"   Large tx threshold:            ${large_tx_threshold:,.2f}")
    
    # Volatility
    baseline_std = baseline_data.groupby('Day')['amount'].std().mean()
    crisis_std = crisis_data.groupby('Day')['amount'].std().mean()
    
    print(f"\n⚡ MARKET STRESS (Transaction Volatility):")
    print(f"   Baseline volatility:           ${baseline_std:,.0f}")
    print(f"   Crisis volatility:             ${crisis_std:,.0f}")
    print(f"   Increase:                      {(crisis_std/baseline_std - 1)*100:.1f}%")
    
    # First warning signal
    vol_spike_threshold = baseline_vol * 1.5
    hourly_baseline_vol = baseline_vol / 24  # Convert daily to hourly
    first_spike_hourly = hourly[hourly['Total_Volume'] > hourly_baseline_vol * 1.5]
    
    if len(first_spike_hourly) > 0:
        first_spike = first_spike_hourly['Hour'].min()
        days_before_peak = (peak_hour - first_spike).total_seconds() / 86400
        
        print(f"\n🚨 EARLY WARNING SIGNALS:")
        print(f"   First 1.5x volume spike:       {first_spike}")
        print(f"   Days before peak:              {days_before_peak:.1f} days")
    
    # Network concentration
    baseline_days = daily[daily['Day'] < pd.to_datetime(CRISIS_START)]
    crisis_days = daily[
        (daily['Day'] >= pd.to_datetime(CRISIS_START)) & 
        (daily['Day'] <= pd.to_datetime(CRISIS_END))
    ]
    
    if len(baseline_days) > 0 and len(crisis_days) > 0:
        baseline_hhi = baseline_days['HHI_Senders'].mean()
        crisis_hhi = crisis_days['HHI_Senders'].mean()
        
        print(f"\n🌐 NETWORK CONCENTRATION:")
        print(f"   Baseline HHI:                  {baseline_hhi:.4f}")
        print(f"   Crisis HHI:                    {crisis_hhi:.4f}")
        print(f"   Change:                        {(crisis_hhi/baseline_hhi - 1)*100:+.1f}%")
        
        if crisis_hhi > baseline_hhi:
            print(f"   Interpretation:                More concentrated (fewer large players)")
        else:
            print(f"   Interpretation:                More distributed (many participants)")
    
    # Summary interpretation
    print("\n" + "="*80)
    print("INTERPRETATION FOR YOUR SLIDE DECK:")
    print("="*80)
    print(f"""
1. CRISIS ONSET:
   - Confidence began breaking when volume exceeded baseline by {(crisis_vol/baseline_vol - 1)*100:.0f}%
   - First warning signal appeared on {first_spike if 'first_spike' in locals() else 'early May'}
   - Transaction count surged {(crisis_count/baseline_count - 1)*100:.0f}%, indicating broad panic

2. PANIC PROPAGATION:
   - Large transactions increased {(crisis_large/baseline_large - 1)*100:.0f}%
   - This suggests {'sophisticated actors detected instability first' if crisis_large/baseline_large > crisis_count/baseline_count else 'panic was widespread from the start'}
   - Network became {'more concentrated' if crisis_hhi > baseline_hhi else 'more distributed'} during crisis

3. STRESS MANIFESTATION:
   - Transaction volatility increased {(crisis_std/baseline_std - 1)*100:.0f}%
   - Peak activity: ${peak_vol_hourly:,.0f}/hour on {peak_hour.date()}
   - Sustained stress: {stress_hours} hours of elevated activity
    """)

else:
    print("\n⚠️  Insufficient data in baseline or crisis period for comparison")
    print(f"   Baseline transactions: {len(baseline_data):,}")
    print(f"   Crisis transactions: {len(crisis_data):,}")


# ============================================================================
# EXPORT DATA FOR VISUALIZATION
# ============================================================================

print("\n" + "="*80)
print("EXPORTING PROCESSED DATA")
print("="*80)

# Export to CSV for use in visualizations
hourly.to_csv('hourly_metrics.csv', index=False)
daily.to_csv('daily_metrics.csv', index=False)

# Export key events
key_events = pd.DataFrame([
    {'Date': pd.to_datetime(CRISIS_START), 'Event': 'UST Depeg Begins (May 7)'},
    {'Date': pd.to_datetime('2022-05-08'), 'Event': 'Confidence Collapses (May 8)'},
    {'Date': pd.to_datetime('2022-05-09'), 'Event': 'LUNA Death Spiral (May 9)'},
    {'Date': pd.to_datetime('2022-05-10'), 'Event': 'Near-Total Collapse (May 10)'},
    {'Date': pd.to_datetime(CRISIS_END), 'Event': 'Crisis Containment (May 12+)'}
])
if 'first_spike' in locals():
    key_events = pd.concat([
        pd.DataFrame([{'Date': first_spike, 'Event': 'First Warning Signal'}]),
        key_events
    ]).sort_values('Date').reset_index(drop=True)

key_events.to_csv('key_crisis_events.csv', index=False)

print("✓ Exported: hourly_metrics.csv")
print("✓ Exported: daily_metrics.csv")
print("✓ Exported: key_crisis_events.csv")

print("\n✅ ANALYSIS COMPLETE!")
print("\nProcessed data is ready for visualization.")
print("Load 'hourly_metrics.csv' into your Visualiser class.")


SECTION A, QUESTION 1: KEY FINDINGS
When the Peg Breaks: Onset and Spread

📊 VOLUME DYNAMICS:
   Baseline avg daily volume:     $118,246,948
   Crisis avg daily volume:       $719,822,766
   Increase:                      508.7%
   Peak hourly volume:            $407,722,411
   Peak occurred:                 2022-05-10 07:00:00

🔄 TRANSACTION ACTIVITY:
   Baseline avg daily txs:        1460
   Crisis avg daily txs:          5256
   Increase:                      259.9%
   Peak hourly txs:               1,628
   Peak occurred:                 2022-05-10 22:00:00

🐋 WHALE ACTIVITY (>95th percentile):
   Baseline avg daily large txs:  74.5
   Crisis avg daily large txs:    294.1
   Increase:                      294.5%
   Large tx threshold:            $360,788.72

⚡ MARKET STRESS (Transaction Volatility):
   Baseline volatility:           $408,279
   Crisis volatility:             $889,012
   Increase:                      117.7%

🚨 EARLY WARNING SIGNALS:
   First 1.5x volume spike:    

In [69]:
from src.Visualiser import Visualiser
import pandas as pd

# Load processed data
hourly_df = pd.read_csv('hourly_metrics.csv')
hourly_df['Hour'] = pd.to_datetime(hourly_df['Hour'])
hourly_df['Date'] = hourly_df['Hour']

# PRE-CALCULATE transformed columns (don't use lambdas)
hourly_df['Total_Volume_M'] = hourly_df['Total_Volume'] / 10000000
hourly_df['Tx_Count_K'] = hourly_df['Tx_Count'] / 100
hourly_df['Avg_Tx_K'] = hourly_df['Avg_Tx']

# Initialize visualiser
vis = Visualiser(hourly_df)
vis.focus(CRISIS_START,CRISIS_END)

# Use the pre-calculated columns directly (no lambdas)
graphs = [
    {
        "Title": "Crisis Onset: Volume and Transaction Surge",
        "Traces": [
            {
                "Name": "Total Volume (100M)",
                "x": "Date",
                "y": "Total_Volume_M",  # Direct column name
                "Color": "crimson"
            },
            {
                "Name": "Transaction Count (thousands)",
                "x": "Date",
                "y": "Tx_Count_K",  # Direct column name
                "Color": "steelblue"
            }
        ]
    },
    {
        "Title": "Whale Exodus: Large Transaction Activity",
        "Traces": [
            {
                "Name": "Average Tx Size",
                "x": "Date",
                "y": "Avg_Tx_K",  # Direct column name
                "Color": "darkorange"
            },
            {
                "Name": "Large Tx Count",
                "x": "Date",
                "y": "Large_Tx_Count",  # This was already a direct column
                "Color": "darkred"
            }
        ]
    }
]

vis.plot_indicators(
    title="Terra-Luna Crisis: Confidence Breakdown Analysis",
    graphs=graphs
)

In [86]:
import importlib
import src.Visualiser as V

importlib.reload(V)
from src.Visualiser import Visualiser

# Load processed data
daily_df = pd.read_csv('daily_metrics.csv')
daily_df['Day'] = pd.to_datetime(daily_df['Day'])
daily_df['Date'] = daily_df['Day']

# PRE-CALCULATE transformed columns (don't use lambdas)
daily_df['Total_Volume_M'] = daily_df['Daily_Volume'] / 100000000
daily_df['Tx_Count_K'] = daily_df['Tx_Count'] /1000

# Initialize visualiser
vis = Visualiser(daily_df)
vis.focus(CRISIS_START,CRISIS_END)

# Use the pre-calculated columns directly (no lambdas)
graphs = [
    {
        "Title": "Crisis Onset: Volume and Transaction Surge",
        "Traces": [
            {
                "Name": "Total Volume of Tokens / 100M",
                "x": "Date",
                "y": "Total_Volume_M",  # Direct column name
                "Color": "crimson"
            },
            {
                "Name": "Transaction Count / Thousands",
                "x": "Date",
                "y": "Tx_Count_K",  # Direct column name
                "Color": "steelblue"
            }            
        ]
    }
]

vis.plot_indicators(
    title="Terra-Luna Crisis: Confidence Breakdown Analysis",
    graphs=graphs
)